In [ ]:
# eda_neuralsoc.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE_PATH = "/content/drive/MyDrive/CICIDS2017/Monday-WorkingHours.pcap_ISCX.csv"  # change per file
SAMPLE_SIZE = 50000   # use None to load full file (risky on Colab)
TARGET_COL  = " Label"  # CICIDS columns have a leading space — watch this!
# ────────────────────────────────────────────────────────────────────────────

# 1. LOAD
df = pd.read_csv(FILE_PATH, nrows=SAMPLE_SIZE)
df.columns = df.columns.str.strip()  # strip whitespace from ALL column names
print(f"Shape: {df.shape}")
print(f"\nLabel distribution:\n{df['Label'].value_counts()}")

# 2. BASIC INFO
print("\n── Dtypes & Nulls ──")
info = pd.DataFrame({
    'dtype'   : df.dtypes,
    'nulls'   : df.isnull().sum(),
    'null_%'  : (df.isnull().sum() / len(df) * 100).round(2),
    'nunique' : df.nunique()
})
print(info[info['nulls'] > 0])  # only show problematic cols

# 3. INFINITY / BAD VALUES (CICIDS is notorious for these)
num_df = df.select_dtypes(include=[np.number])
inf_cols = num_df.columns[np.isinf(num_df).any()]
print(f"\nColumns with Inf values: {list(inf_cols)}")

# Replace inf with NaN so we can handle uniformly
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
print(f"Shape after cleaning: {df.shape}")

# 4. CLASS DISTRIBUTION PLOT
plt.figure(figsize=(10, 4))
df['Label'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('class_dist.png', dpi=150)
plt.show()

# 5. FEATURE STATS (numeric only)
stats = df.select_dtypes(include=[np.number]).describe().T
stats['cv'] = stats['std'] / (stats['mean'].abs() + 1e-9)  # coefficient of variation
print("\n── Top 10 High-Variance Features ──")
print(stats.sort_values('cv', ascending=False).head(10)[['mean','std','cv']])

# 6. CORRELATION HEATMAP (top 20 features by variance)
top_feats = df.select_dtypes(include=[np.number]).var().sort_values(ascending=False).head(20).index
plt.figure(figsize=(14, 10))
sns.heatmap(df[top_feats].corr(), cmap='coolwarm', center=0,
            linewidths=0.3, annot=False)
plt.title('Correlation Heatmap (Top 20 Variance Features)')
plt.tight_layout()
plt.savefig('corr_heatmap.png', dpi=150)
plt.show()

# 7. DROP HIGHLY CORRELATED FEATURES (threshold 0.95)
corr_matrix = df[top_feats].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f"\nHighly correlated features to drop: {to_drop}")

# 8. ZERO/NEAR-ZERO VARIANCE FEATURES
num_cols = df.select_dtypes(include=[np.number]).columns
low_var = df[num_cols].var()[df[num_cols].var() < 1e-5].index.tolist()
print(f"Near-zero variance features: {low_var}")

# 9. ENCODE LABEL → INTEGER
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['Label'])
print(f"\nLabel mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 10. FINAL FEATURE SET RECOMMENDATION
drop_all = list(set(to_drop + low_var + ['Label', 'label_enc']))
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_all]
print(f"\nRecommended feature count for training: {len(feature_cols)}")
print(feature_cols)

# 11. SAVE CLEAN SAMPLE (optional — for quick model prototyping)
df[feature_cols + ['label_enc']].to_csv('clean_sample.csv', index=False)
print("\nSaved clean_sample.csv ✅")